# 1. Introduction and Real-World Decision Problem
## 1.1 Das reale Szenario
Als überregionaler Logistikdienstleister (Decision Maker) betreiben wir ein multimodales Transportnetzwerk. Dieses Netzwerk besteht aus verschiedenen Standorten (Städten und Logistik-Terminals), die über feste Transportwege für verschiedene Verkehrsträger (Straße, Schiene, Luft) miteinander verbunden sind.

## 1.2 Der operative Prozess
Unsere Kunden übergeben uns täglich eine Vielzahl von Sendungen. Jede Sendung hat spezifische Eigenschaften: einen festgelegten Start- und Zielort, ein bestimmtes Volumen/Gewicht, einen Frachttyp (z. B. Standardfracht oder Gefahrgut wie Batterien) sowie eine strikte Lieferfrist. Unser operativer Prozess sieht vor, diese Sendungen durch unser Netzwerk zu navigieren. Dabei haben wir die Möglichkeit der Konsolidierung: Anstatt jede Sendung einzeln direkt ans Ziel zu fahren, können wir Sendungen an Transfer-Terminals umladen und bündeln. So können beispielsweise viele kleine Pakete, die per Lkw am Terminal ankommen, gemeinsam auf einen ressourceneffizienten Güterzug oder in ein Frachtflugzeug umgeladen werden.

## 1.3 Ressourcen, Limitierungen und Kostenstrukturen
In der Realität ist dieser Prozess durch knappe Ressourcen und strikte Regeln eingeschränkt:

- Flottenkapazitäten und Fixkosten: Wir können nicht unendlich viel Ware auf einer Strecke transportieren. Unsere verfügbaren Lkw-Flotten, die gebuchten Waggons auf der Schiene und die reservierten Frachträume in Flugzeugen haben ein maximales Transportvolumen. Zudem fallen für reservierte Transportmittel (z.B. das Buchen eines kompletten Waggons) Fixkosten an, unabhängig davon, ob die Kapazität voll ausgeschöpft wird. Dies zwingt unser Netzwerk zu intelligenter Konsolidierung.
- Explizite Terminal-Prozesse: Terminals haben nicht nur eine maximale tägliche Abfertigungskapazität
. Das Umladen erfordert zudem konkrete Lade-, Entlade- und Transferzeiten, die in die Berechnung der Gesamtreisezeit jeder Sendung exakt einfließen müssen.
- Zeitliche Restriktionen: Jede Sendung muss ihr Ziel erreichen, bevor ihre individuelle Deadline abläuft
. Ein Wechsel auf einen günstigeren Transportmodus ist nur zulässig, wenn die Summe aus Fahrzeiten und den expliziten Terminal-Prozessen das Zeitfenster nicht sprengt.
- Frachtspezifische Restriktionen: Bestimmte Güter unterliegen Transportverboten. Beispielsweise dürfen Batterien aus Sicherheitsgründen nicht per Luftfracht transportiert werden

## 1.4 Die Entscheidungsfrage
Das formulierte Entscheidungsproblem lautet:
> Welche exakten Routen und Transportmittel (Straße, Schiene, Luft) sollten für eine gegebene Menge an Sendungen gewählt werden, um eine gewichtete Kombination aus Transportkosten (inklusive Kapazitäts-Fixkosten) und CO₂-Emissionen zu minimieren, während gleichzeitig Flottenkapazitäten, zeitaufwändige Terminal-Prozesse, Lieferfristen und frachtspezifische Transportverbote eingehalten werden?


## 1.5 Future Scope
Um das Modell perspektivisch für Logistikunternehmen noch realitätsnaher zu gestalten, können je nach Rechenkapazität des Solvers folgende Dimensionen im weiteren Projektverlauf ergänzt werden:
- Soft Time Windows: Anstelle von strikten Deadlines erhalten Sendungen Ziel-Zeitfenster. Wenn ein Paket zu spät (oder zu früh) ankommt, wird das Modell nicht sofort unzulässig, sondern es fallen proportionale Strafkosten an.
- Verschiedene Fahrzeugtypen: Die Straßen-Infrastruktur wird durch spezifische Fahrzeugtypen (z. B. kleine Lieferwagen, schwere Diesel-Lkws, emissionsfreie Elektro-Lkws) mit unterschiedlichen Kapazitäten, Kosten und spezifischen CO₂-Emissionsprofilen abgebildet.
- Fahrerrestriktionen: Die Berücksichtigung von gesetzlichen Lenk- und Ruhezeiten auf der Straße. Dies würde den Lkw-Transport auf Langstrecken im Vergleich zur Schiene zeitlich deutlich verlangsamen.

## 2. Modellstruktur: Hubs, Nodes und zeitabhängige Arcs

Das Transportnetzwerk wird als gerichteter Graph modelliert. Physische Orte wie Berlin, Hamburg, Frankfurt oder München werden als **Hubs** bezeichnet. Da ein Hub mehrere Verkehrsträger unterstützen kann, wird jeder Hub zusätzlich in mode-spezifische Nodes aufgeteilt, z. B. `BER_road`, `BER_rail` oder `FRA_air`.

Für die zeitliche Modellierung wird daraus ein **time-expanded network** erzeugt. Ein Node beschreibt dann nicht nur Ort und Verkehrsträger, sondern auch einen konkreten Zeitpunkt:

- `BER_road_08`: Straße-Terminal in Berlin um 08:00 Uhr
- `BER_rail_10`: Schienen-Terminal in Berlin um 10:00 Uhr
- `HAM_road_12`: Straße-Terminal in Hamburg um 12:00 Uhr

Die Kanten des Graphen heißen **Arcs**. Ein Arc verbindet zwei Zeit-Nodes und bildet entweder eine Transportbewegung, einen Terminaltransfer oder Warten am Hub ab:

- **Transport-Arc:** Bewegung zwischen zwei Hubs, z. B. `BER_road_08 -> HAM_road_12`
- **Transfer-Arc:** Wechsel des Verkehrsträgers innerhalb eines Hubs, z. B. `BER_road_08 -> BER_rail_10`
- **Waiting-Arc:** Warten oder Lagern am selben Hub, z. B. `BER_road_08 -> BER_road_09`

Jeder Arc enthält Parameter wie Dauer, Kapazität, Kosten und Emissionen. Transportkosten und Emissionen können aus Distanz und mode-spezifischen Faktoren berechnet werden, z. B. `cost_per_ton_km` und `emissions_kg_per_ton_km`. Transfer- und Waiting-Arcs können eigene Kosten, Zeiten und Kapazitäten besitzen.

Fahrpläne werden pro Hub definiert. Daraus werden anschließend automatisch zeitabhängige Arcs generiert. Straße kann im einfachen Modell stündlich verfügbar sein, während Schiene und Luft feste Abfahrtszeiten erhalten. Zeitabhängige Kosten, z. B. höhere LKW-Kosten nachts, können über einen Kostenmultiplikator auf den jeweiligen Arc abgebildet werden.

Im MILP entscheidet die Variable `x[shipment, arc]`, ob eine Sendung einen bestimmten Arc nutzt. Die Flusserhaltung stellt sicher, dass eine Sendung zeitlich konsistent durch das Netzwerk läuft. Deadlines werden eingehalten, indem Sendungen nur Ziel-Zeit-Nodes erreichen dürfen, die vor ihrer Deadline liegen. Kapazitätsrestriktionen begrenzen die Summe der transportierten Gewichte auf jedem Arc.


In [ ]:
hubs = [
    {"id": "BER", "name": "Berlin", "lat": 52.5200, "lon": 13.4050},
    {"id": "HAM", "name": "Hamburg", "lat": 53.5511, "lon": 9.9937},
    {"id": "FRA", "name": "Frankfurt", "lat": 50.1109, "lon": 8.6821},
    {"id": "MUC", "name": "Munich", "lat": 48.1351, "lon": 11.5820},
]

hub_time_plans = {
    "BER": {
        "opening_hours": {
            "road": list(range(0, 24)),  # LKW jederzeit
            "rail": [5, 6, 7, 8, 17, 18],  # Terminalfenster
            "air": [],
        },
        "transfer_windows": {
            ("road", "rail"): [4, 5, 6, 16, 17],
            ("rail", "road"): [9, 10, 21, 22],
        },
        "departures": {
            "road": list(range(0, 24)),  # stündlich möglich
            "rail": [6, 18],  # Züge ab BER
            "air": [],
        },
        "cost_multipliers": {
            "road": {
                "day": 1.0,
                "night": 1.25,
            }
        },
    },
    "FRA": {
        "opening_hours": {
            "road": list(range(0, 24)),
            "rail": [6, 7, 8, 18, 19],
            "air": [7, 8, 9, 15, 16],
        },
        "transfer_windows": {
            ("road", "rail"): [5, 6, 17],
            ("road", "air"): [6, 7, 14, 15],
            ("air", "road"): [10, 11, 18],
        },
        "departures": {
            "road": list(range(0, 24)),
            "rail": [7, 19],
            "air": [8, 16],
        },
        "cost_multipliers": {
            "road": {
                "day": 1.0,
                "night": 1.3,
            }
        },
    },
}

nodes = [
    {"id": "BER_road", "hub": "BER", "mode": "road"},
    {"id": "BER_rail", "hub": "BER", "mode": "rail"},
    {"id": "HAM_road", "hub": "HAM", "mode": "road"},
    {"id": "HAM_rail", "hub": "HAM", "mode": "rail"},
    {"id": "FRA_road", "hub": "FRA", "mode": "road"},
    {"id": "FRA_rail", "hub": "FRA", "mode": "rail"},
    {"id": "FRA_air", "hub": "FRA", "mode": "air"},
    {"id": "MUC_road", "hub": "MUC", "mode": "road"},
    {"id": "MUC_rail", "hub": "MUC", "mode": "rail"},
    {"id": "MUC_air", "hub": "MUC", "mode": "air"},
]

mode_factors = {
    "road": {"cost_per_ton_km": 1.20, "emissions_kg_per_ton_km": 0.09},
    "rail": {"cost_per_ton_km": 0.70, "emissions_kg_per_ton_km": 0.025},
    "air": {"cost_per_ton_km": 3.50, "emissions_kg_per_ton_km": 0.60},
}

arc_templates = [
    {
        "id": "BER_road__HAM_road",
        "from_hub": "BER",
        "to_hub": "HAM",
        "mode": "road",
        "distance_km": 290,
        "duration_h": 4,
        "capacity_ton": 30,
    },
    {
        "id": "BER_rail__HAM_rail",
        "from_hub": "BER",
        "to_hub": "HAM",
        "mode": "rail",
        "distance_km": 285,
        "duration_h": 4,
        "capacity_ton": 120,
    },
]

In [ ]:
time_arcs = []

for template in arc_templates:
    from_hub = template["from_hub"]
    mode = template["mode"]

    for departure_h in hub_time_plans[from_hub]["departures"][mode]:
        arrival_h = departure_h + template["duration_h"]

        if arrival_h <= 24:
            time_arcs.append(
                {
                    "id": f"{template['id']}__dep_{departure_h}",
                    "from_node": f"{from_hub}_{mode}_{departure_h}",
                    "to_node": f"{template['to_hub']}_{mode}_{arrival_h}",
                    "arc_type": "transport",
                    "mode": mode,
                    "departure_h": departure_h,
                    "arrival_h": arrival_h,
                    "duration_h": template["duration_h"],
                    "distance_km": template["distance_km"],
                    "capacity_ton": template["capacity_ton"],
                }
            )